
# Thermodynamics Laboratory — Interactive Version

This notebook is a **macroscopic thermodynamics simulator**.

You will not see particles. Instead, you control macroscopic variables and observe
how the thermodynamic state changes.

The first interactive apparatus is a gas in a cylinder with a movable piston.

You can choose:

- number of particles \(N\)
- initial temperature \(T_i\)
- initial volume \(V_i\)
- final volume \(V_f\)
- process type: **adiabatic** or **isothermal**

The notebook then calculates the process and displays

\[
P(V),\qquad T(V),\qquad E(V),\qquad W.
\]

The sign convention is

\[
\delta W=-P_{\rm ext}\,dV,
\]

so \(W>0\) means work is done **on** the system.


In [3]:

# This cell checks that the interactive widget machinery is available.
# In a normal Jupyter installation, ipywidgets is usually already installed.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

print("ipywidgets version:", widgets.__version__)
print("Interactive controls will appear below after executing the next cell.")


ModuleNotFoundError: No module named 'ipywidgets'


## Interactive Experiment 1 — A piston

**Prediction first:** Before changing the sliders, predict what should happen to
the temperature when you compress an adiabatic gas.

Then use the controls below.

> **Important:** The sliders are live Jupyter widgets. They are not part of the
> static notebook preview. To use them, open this file in Jupyter Notebook or
> JupyterLab and execute the cells. If you are viewing the notebook in a web
> preview that does not execute widgets, use JupyterLab or Voilà.


In [2]:

# Thermodynamic model
k_B = 1.0
C_V = 1.5 * k_B
gamma = 1.0 + k_B / C_V

def process_piston(N, T_initial, V_initial, V_final, process="Adiabatic", n=300):
    if V_final <= 0 or V_initial <= 0:
        raise ValueError("Volumes must be positive.")

    V = np.linspace(V_initial, V_final, n)

    if process == "Adiabatic":
        T = T_initial * (V_initial / V)**(gamma - 1.0)
    elif process == "Isothermal":
        T = np.full_like(V, T_initial)
    else:
        raise ValueError("Unknown process type.")

    E = C_V * N * T
    P = N * k_B * T / V

    dV = np.diff(V)
    P_mid = 0.5 * (P[:-1] + P[1:])
    dW = -P_mid * dV
    W = np.concatenate([[0.0], np.cumsum(dW)])

    return pd.DataFrame({"V": V, "T": T, "P": P, "E": E, "W": W})

def run_piston(N=100, T_initial=1.0, V_initial=10.0, V_final=5.0,
               process="Adiabatic"):
    df = process_piston(N, T_initial, V_initial, V_final, process)

    E0 = df.E.iloc[0]
    Ef = df.E.iloc[-1]
    W = df.W.iloc[-1]
    Q = (Ef - E0) - W

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(df.V, df.P)
    axes[0].set_xlabel("$V$")
    axes[0].set_ylabel("$P$")
    axes[0].set_title("$P$-$V$ path")
    axes[0].invert_xaxis()

    axes[1].plot(df.V, df["T"])
    axes[1].set_xlabel("$V$")
    axes[1].set_ylabel("$T$")
    axes[1].set_title("Temperature")
    axes[1].invert_xaxis()

    axes[2].plot(df.V, df.E)
    axes[2].set_xlabel("$V$")
    axes[2].set_ylabel("$E$")
    axes[2].set_title("Internal energy")
    axes[2].invert_xaxis()

    plt.tight_layout()
    plt.show()

    summary = pd.DataFrame({
        "Quantity": ["Initial E", "Final E", "Delta E", "Q", "W", "Q + W",
                     "First-law residual"],
        "Value": [E0, Ef, Ef-E0, Q, W, Q+W, (Ef-E0)-(Q+W)]
    })

    display(summary)

# Controls
N = widgets.IntSlider(
    value=100, min=10, max=500, step=10,
    description="N", continuous_update=False,
    style={"description_width": "120px"}
)
T_initial = widgets.FloatSlider(
    value=1.0, min=0.2, max=3.0, step=0.1,
    description="Initial T", continuous_update=False,
    style={"description_width": "120px"}
)
V_initial = widgets.FloatSlider(
    value=10.0, min=2.0, max=20.0, step=0.5,
    description="Initial V", continuous_update=False,
    style={"description_width": "120px"}
)
V_final = widgets.FloatSlider(
    value=5.0, min=1.0, max=20.0, step=0.5,
    description="Final V", continuous_update=False,
    style={"description_width": "120px"}
)
process = widgets.ToggleButtons(
    options=["Adiabatic", "Isothermal"],
    value="Adiabatic",
    description="Process",
    style={"description_width": "120px"}
)

controls = widgets.VBox([N, T_initial, V_initial, V_final, process])

interactive_plot = widgets.interactive_output(
    run_piston,
    {
        "N": N,
        "T_initial": T_initial,
        "V_initial": V_initial,
        "V_final": V_final,
        "process": process
    }
)

display(controls)
display(interactive_plot)


NameError: name 'widgets' is not defined


### Questions

1. For the adiabatic process, what happens to \(T\) when \(V\) decreases?
2. For the isothermal process, what changes and what remains constant?
3. Compare the work \(W\) for the two paths.
4. In each case, verify
   \[
   \Delta E=Q+W.
   \]
5. Why can the same initial and final volumes have different values of \(Q\) and \(W\)?



# Interactive Experiment 2 — Two systems exchanging heat

Two macroscopic systems are placed in thermal contact.

You control

\[
N_1,\;N_2,\;T_1,\;T_2.
\]

The total energy is fixed, and the systems exchange energy until

\[
T_1=T_2.
\]

The notebook shows the temperature trajectories and the total entropy

\[
S_{\rm total}=S_1+S_2.
\]

This is the computational experiment corresponding to the entropy/extremum discussion
in Lecture 3.


In [ ]:

def temperature_from_E(E, N):
    return E / (C_V * N)

def entropy_ideal(E, V, N):
    T = temperature_from_E(E, N)
    return C_V * N * np.log(T) + N * k_B * np.log(V)

def thermal_exchange(N1, N2, T1_initial, T2_initial, V1=10.0, V2=10.0,
                     n=250, rate=5.0):
    E1_0 = C_V * N1 * T1_initial
    E2_0 = C_V * N2 * T2_initial
    E_total = E1_0 + E2_0

    T_eq = E_total / (C_V * (N1 + N2))
    E1_eq = C_V * N1 * T_eq

    t = np.linspace(0, 1, n)
    E1 = E1_eq + (E1_0 - E1_eq) * np.exp(-rate*t)
    E2 = E_total - E1
    T1 = temperature_from_E(E1, N1)
    T2 = temperature_from_E(E2, N2)
    S_total = entropy_ideal(E1, V1, N1) + entropy_ideal(E2, V2, N2)

    return pd.DataFrame({
        "t": t, "E1": E1, "E2": E2,
        "T1": T1, "T2": T2, "S_total": S_total
    })

def run_thermal(N1=100, N2=100, T1_initial=2.0, T2_initial=0.5):
    df = thermal_exchange(N1, N2, T1_initial, T2_initial)

    fig1, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(df.t, df.T1, label="$T_1$")
    ax1.plot(df.t, df.T2, label="$T_2$")
    ax1.set_xlabel("Scaled time")
    ax1.set_ylabel("Temperature")
    ax1.set_title("Thermal equilibration")
    ax1.legend()
    plt.show()

    fig2, ax2 = plt.subplots(figsize=(7, 4))
    ax2.plot(df.t, df.S_total)
    ax2.set_xlabel("Scaled time")
    ax2.set_ylabel("$S_{\\rm total}$")
    ax2.set_title("Total entropy")
    plt.show()

    T_eq = df.T1.iloc[-1]
    display(pd.DataFrame({
        "Quantity": [
            "Initial T1", "Initial T2",
            "Final T1", "Final T2",
            "Equilibrium T", "Delta S_total"
        ],
        "Value": [
            df.T1.iloc[0], df.T2.iloc[0],
            df.T1.iloc[-1], df.T2.iloc[-1],
            T_eq, df.S_total.iloc[-1] - df.S_total.iloc[0]
        ]
    }))

N1 = widgets.IntSlider(value=100, min=10, max=500, step=10,
                       description="N1", continuous_update=False)
N2 = widgets.IntSlider(value=100, min=10, max=500, step=10,
                       description="N2", continuous_update=False)
T1 = widgets.FloatSlider(value=2.0, min=0.2, max=3.0, step=0.1,
                         description="Initial T1", continuous_update=False)
T2 = widgets.FloatSlider(value=0.5, min=0.2, max=3.0, step=0.1,
                         description="Initial T2", continuous_update=False)

thermal_controls = widgets.VBox([N1, N2, T1, T2])
thermal_output = widgets.interactive_output(
    run_thermal,
    {"N1": N1, "N2": N2, "T1_initial": T1, "T2_initial": T2}
)

display(thermal_controls)
display(thermal_output)



## 12. A student-designed experiment

Use the controls to formulate and test your own hypothesis.

For example:

> **Hypothesis:** Increasing \(N_1\) while keeping the two initial temperatures fixed
> changes the final equilibrium temperature.

Write down your prediction, run the experiment, and explain the result using the
thermodynamic state variables.

### Recommended final challenge

Design an experiment that demonstrates all three statements:

\[
\boxed{
\Delta E \text{ is path independent},
\qquad
Q \text{ is path dependent},
\qquad
W \text{ is path dependent}.
}
\]

Then design a separate experiment showing

\[
\boxed{
S_{\rm total}\text{ increases toward equilibrium}.
}
\]



## Running the interactive notebook

The interactive controls require a live Jupyter environment with `ipywidgets`.

From a terminal, a typical setup is:

```bash
pip install jupyterlab ipywidgets matplotlib pandas numpy
jupyter lab
```

Then open this notebook and execute the cells.

**JupyterLab should display the sliders and buttons directly below the code cell.**

For teaching, Voilà is another good option because it turns the notebook into a clean
browser-based laboratory while preserving the widgets:

```bash
pip install voila
voila Thermodynamics_Laboratory_Interactive_v2.ipynb
```
